# Оценка моделей на своей разметке

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SergeyShk/Speech-to-Text-Russian/blob/master/notebooks/04_model_evaluation.ipynb)

Как выбрать модель для своих записей командой `stt-ru evaluate`: разметка CSV, доля ошибок слов (WER) и символов (CER), скорость, нормализация текста, числа цифрами и словами, разбор записей с наибольшими ошибками и квантизация весов.

> **Важно.** Этот ноутбук показывает, как пользоваться оценкой, а не сравнивает модели. Пример - 100 фраз из FLEURS: это чтение вслух в тишине, и на звонках, лекциях или диктовке модели расставятся иначе. Выбирать модель стоит по своим записям.

## Установка

В Colab ячейка ставит stt-ru и `pandas`, `pyarrow`; в окружении, где пакет уже есть, ничего не делает.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("stt_ru") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "stt-ru[ffmpeg,eval]", "pandas", "pyarrow"],
        check=True,
    )

## Разметка

[FLEURS](https://huggingface.co/datasets/google/fleurs) (Google, CC BY 4.0) - короткие фразы, прочитанные носителями языка, с эталонным текстом. Ячейка скачивает русскую валидационную часть (около 250 МБ, один раз) и сохраняет первые 100 фраз как WAV и разметку CSV с колонками `audio` и `text`, как для своих записей.

In [2]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

from stt_ru import Transcriber
from stt_ru.evaluation import evaluate, format_table, load_manifest, normalize

pd.set_option("display.max_colwidth", None)

DATA = Path("data") / "fleurs"
DATA.mkdir(parents=True, exist_ok=True)
parquet = hf_hub_download(
    "google/fleurs",
    "ru_ru/validation/0000.parquet",
    repo_type="dataset",
    revision="refs/convert/parquet",
)
table = pq.read_table(parquet, columns=["audio", "raw_transcription"]).slice(0, 100)
rows = []
texts = table["raw_transcription"].to_pylist()
for audio, text in zip(table["audio"].to_pylist(), texts, strict=True):
    path = DATA / audio["path"]
    path.write_bytes(audio["bytes"])
    rows.append({"audio": path.name, "text": " ".join(text.split())})
references = pd.DataFrame(rows)
references.to_csv(DATA / "eval.csv", index=False)
references.head(3)

,audio,text
0,10005687533826592442.wav,"Они умеют отлично видеть в темноте при помощи ночного видения и почти незаметно передвигаться. Оцелоты выслеживают добычу, сливаясь с окружающей обстановкой, а затем набрасываются на добычу."
1,10042203520901241191.wav,"Он сказал, что создал дверной звонок, работающий от WiFi."
2,10095490839809792710.wav,"В Японии приблизительно 7000 островов (самый большой из которых — Хонсю), что делает Японию 7-м по величине островом в мире!"


## Сравнение моделей

`stt-ru evaluate` распознает каждую запись каждой моделью и печатает таблицу: WER и CER по всем записям вместе, скорость относительно реального времени и число записей с ошибкой. `-o` сохраняет подробный отчет в JSON. Модели скачиваются при первом запуске; у T-one нет весов int8, поэтому для нее загружаются исходные веса. Предупреждение об этом и полосы загрузки моделей в выводе команды скрыты.

In [3]:
import os

os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [4]:
!stt-ru evaluate data/fleurs/eval.csv -q -o data/fleurs/report.json -m gigaam-v3-e2e-rnnt -m gigaam-v3-rnnt -m t-tech/t-one -m alphacep/vosk-model-small-ru

Модель                        WER, %  CER, %  Скорость  Записей  Ошибок
gigaam-v3-e2e-rnnt               6.2     2.1     66.5x      100       0
gigaam-v3-rnnt                   7.4     4.4     65.8x      100       0
t-tech/t-one                    13.2     5.5     34.2x      100       0
alphacep/vosk-model-small-ru    12.5     5.9    111.0x      100       0
gigaam-v3-e2e-rnnt: в 6 записях цифры есть только в эталоне или только в распознанном тексте - если числа записаны по-разному (цифрами и словами), WER завышен
gigaam-v3-rnnt: в 15 записях цифры есть только в эталоне или только в распознанном тексте - если числа записаны по-разному (цифрами и словами), WER завышен
t-tech/t-one: в 15 записях цифры есть только в эталоне или только в распознанном тексте - если числа записаны по-разному (цифрами и словами), WER завышен
alphacep/vosk-model-small-ru: в 15 записях цифры есть только в эталоне или только в распознанном тексте - если числа записаны по-разному (цифрами и словами), WER завышен


## Числа

Эталон и распознанный текст перед подсчетом нормализуются: нижний регистр, без пунктуации, «ё» как «е». Числа при этом не переводятся: GigaAM-v3 E2E пишет их цифрами, остальные модели здесь - словами. В эталонах FLEURS числа записаны цифрами, поэтому у моделей, которые пишут словами, часть ошибок - разная запись одних и тех же чисел. Об этом и предупреждает команда.

In [5]:
report = json.loads((DATA / "report.json").read_text(encoding="utf-8"))
results = pd.DataFrame(
    [{"model": m["model"], **result} for m in report["models"] for result in m["results"]]
)
results["digits"] = results["reference"].str.contains(r"\d")


def wer(frame):
    errors = frame["substitutions"] + frame["deletions"] + frame["insertions"]
    return errors.sum() / frame["words"].sum()


by_digits = results.groupby(["model", "digits"]).apply(wer, include_groups=False).unstack()
by_digits.columns = ["WER без чисел", "WER с числами"]
print(f"Записей с числами в эталоне: {results.groupby('model')['digits'].sum().iloc[0]} из 100")
by_digits.round(3)

Записей с числами в эталоне: 15 из 100


,WER без чисел,WER с числами
model,,
alphacep/vosk-model-small-ru,0.079,0.313
gigaam-v3-e2e-rnnt,0.039,0.156
gigaam-v3-rnnt,0.027,0.268
t-tech/t-one,0.085,0.321


Нормализация на примере фразы с числами:

In [6]:
sample = references[references["text"].str.contains(r"\d")].iloc[0]["text"]
print(sample)
print(normalize(sample))

В Японии приблизительно 7000 островов (самый большой из которых — Хонсю), что делает Японию 7-м по величине островом в мире!
в японии приблизительно 7000 островов самый большой из которых хонсю что делает японию 7 м по величине островом в мире


## Где модель ошибается

Отчет хранит для каждой записи эталон, распознанный текст и число замен, пропусков и вставок: по нему легко найти записи с наибольшими ошибками и понять их причину.

In [7]:
worst = (
    results[results["model"] == "gigaam-v3-e2e-rnnt"]
    .sort_values("wer", ascending=False)
    .head(5)[["audio", "wer", "reference", "hypothesis"]]
)
worst["audio"] = worst["audio"].map(lambda path: Path(path).name)
worst.round(3)

,audio,wer,reference,hypothesis
49,12022741181094187209.wav,0.304,"6-го октября 1789-го года разозлённая толпа женщин с рынка заставила вернуться в Париж из Версаля пришедших в ужас короля Людовика XVI, королеву Марию-Антуаннету, их двух маленьких детей (11-ти летнюю Марию-Терезу и 4-х летнего Луи-Чарлза) и сестру короля мадам Элизабет.","6 октября 1789 года разъязлённая толпа женщин с рынка заставила вернуться в Париж из Версаля. Пришедших в ужас короля Людовика XIX, королеву Марию Антуанетту, их двух маленьких детей: одиннадцатилетнюю Марию Терезу, четырёхлетнюю Луи Чарльза и сестру короля Мадама Элизабет."
95,14488490255061771869.wav,0.300,"В Японии приблизительно 7000 островов (самый большой из которых — Хонсю), что делает Японию 7-м по величине островом в мире!",В Японии приблизительно 7 000 островов. Самый большой из которых Конь. Что делает Япония? седьмым по величине островом в мире
2,10095490839809792710.wav,0.250,"В Японии приблизительно 7000 островов (самый большой из которых — Хонсю), что делает Японию 7-м по величине островом в мире!",В Японии приблизительно 7 000 островов. Самый большой из которых Контик. Что делает Японию седьмым по величине островом в мире?
7,1024552646813367065.wav,0.231,"В северной части, то есть на хребте Сентинел, находятся самые высокие горы Антарктиды — массив Винсон, самая высшая точка которого достигает 4892 м и называется пик Винсон.","Северной части. То есть на хребте сантинял. Находятся самые высокие горы Антарктиды. Массив Винсен. Самая высшая точка, которого достигает 4 892 м. И называется Пик Винсен."
1,10042203520901241191.wav,0.222,"Он сказал, что создал дверной звонок, работающий от WiFi.","Он сказал, что создал дверной звонок, работающий от Wi-Fi."


## Квантизация

По умолчанию модели загружаются с весами int8: они в несколько раз меньше исходных и обычно быстрее на CPU. Разницу на своих записях показывает та же оценка, здесь - из Python и на маленькой модели Vosk (веса 25 и 90 МБ).

In [8]:
samples = load_manifest(DATA / "eval.csv")
reports = []
for quantization in ("int8", None):
    report = evaluate(
        Transcriber("alphacep/vosk-model-small-ru", quantization=quantization), samples
    )
    report.model += f" ({quantization or 'исходные веса'})"
    reports.append(report)
print(format_table(reports))

Модель                                        WER, %  CER, %  Скорость  Записей  Ошибок
alphacep/vosk-model-small-ru (int8)             12.5     5.9    119.1x      100       0
alphacep/vosk-model-small-ru (исходные веса)    11.8     5.6    102.2x      100       0


## Своя разметка

Для своих записей нужен CSV с колонками `audio` (путь относительно файла разметки) и `text`; для стереозвонков - еще `channel`, номер канала говорящего. Подойдут и JSON Lines с теми же полями, и папка, где у каждой записи есть файл `.txt` с эталоном:

``` text
audio,text,channel
calls/0001.wav,"Добрый день, компания «Ромашка», чем могу помочь?",1
calls/0001.wav,"Здравствуйте, хочу узнать статус заказа.",0
```

Подробнее - на странице [Оценка качества](https://sergeyshk.github.io/Speech-to-Text-Russian/evaluation/) документации.